In [34]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import ollama
import json

In [3]:
df = pd.read_csv("agri01.csv")
df.head()

,crop,state,district,season,year,min_temperature,max_temperature,avg_temperature,rainfall,humidity,...,potassium,irrigation_type,growth_stage,extreme_event,ndvi,area,previous_year_yield,market_price,profit,yield
0,Sugarcane,Uttar Pradesh,Kanpur,Zaid,2022,26.72,39.27,32.72,73.40,42.32,...,43.84,Sprinkler,Sowing,NaN,0.924,4.83,3.62,2709.24,-41538.43,1.95
1,Maize,Tamil Nadu,Coimbatore,Kharif,2017,28.20,38.94,34.02,117.57,40.53,...,128.32,Rain-fed,Flowering,NaN,0.386,9.71,5.55,16493.69,369861.70,2.72
2,Wheat,Punjab,Amritsar,Rabi,2022,14.46,28.22,20.40,89.12,54.95,...,75.04,Drip,Sowing,Flood,0.426,3.91,1.99,24523.69,54185.39,0.85
3,Rice,Tamil Nadu,Madurai,Kharif,2021,17.97,33.07,25.83,191.21,81.22,...,34.42,Drip,Flowering,NaN,0.394,5.63,2.01,23957.64,146010.86,1.32
4,Cotton,Maharashtra,Pune,Rabi,2016,16.63,29.13,21.71,65.82,58.26,...,104.20,Drip,Sowing,NaN,0.382,3.71,5.09,46663.23,393854.06,2.72


In [12]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (10000, 26)

Columns:
['crop', 'state', 'district', 'season', 'year', 'min_temperature', 'max_temperature', 'avg_temperature', 'rainfall', 'humidity', 'soil_type', 'soil_pH', 'organic_matter', 'moisture_content', 'nitrogen', 'phosphorus', 'potassium', 'irrigation_type', 'growth_stage', 'extreme_event', 'ndvi', 'area', 'previous_year_yield', 'market_price', 'profit', 'yield']

Data Types:
crop                    object
state                   object
district                object
season                  object
year                     int64
min_temperature        float64
max_temperature        float64
avg_temperature        float64
rainfall               float64
humidity               float64
soil_type               object
soil_pH                float64
organic_matter         float64
moisture_content       float64
nitrogen               float64
phosphorus             float64
potassium              float64
irrigation_type         object
growth_stage            object
extreme_event

In [13]:
def get_dataset_info(df):

    info = {
        "columns": df.columns.tolist(),
        "data_types": df.dtypes.astype(str).to_dict(),
        "shape": df.shape,
        "missing_values": df.isnull().sum().to_dict(),
        "sample_data": df.head(5).to_dict(orient="records")
    }

    return info

In [7]:
dataset_info = get_dataset_info(df)

dataset_info

{'columns': ['crop',
  'state',
  'district',
  'season',
  'year',
  'min_temperature',
  'max_temperature',
  'avg_temperature',
  'rainfall',
  'humidity',
  'soil_type',
  'soil_pH',
  'organic_matter',
  'moisture_content',
  'nitrogen',
  'phosphorus',
  'potassium',
  'irrigation_type',
  'growth_stage',
  'extreme_event',
  'ndvi',
  'area',
  'previous_year_yield',
  'market_price',
  'profit',
  'yield'],
 'data_types': {'crop': 'object',
  'state': 'object',
  'district': 'object',
  'season': 'object',
  'year': 'int64',
  'min_temperature': 'float64',
  'max_temperature': 'float64',
  'avg_temperature': 'float64',
  'rainfall': 'float64',
  'humidity': 'float64',
  'soil_type': 'object',
  'soil_pH': 'float64',
  'organic_matter': 'float64',
  'moisture_content': 'float64',
  'nitrogen': 'float64',
  'phosphorus': 'float64',
  'potassium': 'float64',
  'irrigation_type': 'object',
  'growth_stage': 'object',
  'extreme_event': 'object',
  'ndvi': 'float64',
  'area': 'floa

In [4]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": " in 50 words"
        }
    ]
)

print(response["message"]["content"])

Donald Trump is a businessman, politician, and reality TV personality. He served as the 45th President of the United States from 2017 to 2021. Trump built his fortune through real estate and entertainment, hosting "The Apprentice" and appearing in films like "Home Alone 2" and "The Fresh Prince of Bel-Air."


In [19]:
import ollama
import json

def generate_pandas_query(question, df):

    dataset_info = get_dataset_info(df)

    prompt = f"""
You are a Pandas query generator.

DATASET INFORMATION:
{json.dumps(dataset_info, default=str)}

USER QUESTION:
{question}

Your task is to generate ONE Pandas expression using the DataFrame df.

STRICT RULES:
1. Return only the Pandas expression.
2. Never generate a function or class.
3. Never generate variable assignments.
4. Never include explanations or comments.
5. Never use Markdown code fences.
6. Use only columns present in the dataset.
7. Do not include the words "Query:" or "Answer:".
8. When using multiple conditions in a Pandas
expression with &, |, or ~ operators, ALWAYS
enclose each individual condition in parentheses.

9. Never combine multiple conditions without
parentheses. Follow Pandas operator precedence rules.

EXAMPLES:

Question: Show people older than 35
Output: df[df["Age"] > 35]

Question: Show people younger than 25
Output: df[df["Age"] < 25]

Question: Calculate average age
Output: df["Age"].mean()

Question: Show the first 5 rows
Output: df.head(5)

Question: Find records where state is Odisha
and year is 2015.

Output:
df[(df["state"] == "Odisha") & (df["year"] == 2015)]

Question: Find records where age is above 25
or salary is above 50000.

Output:
df[(df["age"] > 25) | (df["salary"] > 50000)]

Question: Find people from Odisha who are older
than 25 and have salary above 30000.

Output:
df[(df["state"] == "Odisha") & (df["age"] > 25) & (df["salary"] > 30000)]

Now return only the Pandas expression for the user's question.
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "system",
                "content": "Return only a single Pandas expression. Never generate a function."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0,
            "num_predict": 100
        }
    )

    query = response["message"]["content"].strip()

    # Remove Markdown code fences if the model adds them
    query = query.replace("```python", "").replace("```", "").strip()

    return query

In [20]:
question = "Odisha state in year 2015"

In [21]:
pandas_query = generate_pandas_query(question, df)


In [22]:
pandas_query

'(df["state"] == "Odisha") & (df["year"] == 2015)'

In [23]:
eval(pandas_query)

0       False
1       False
2       False
3       False
4       False
        ...  
9995    False
9996    False
9997    False
9998    False
9999    False
Length: 10000, dtype: bool

In [18]:
df[(df["state"] == "Odisha") & (df["year"] == 2015)]

,crop,state,district,season,year,min_temperature,max_temperature,avg_temperature,rainfall,humidity,...,potassium,irrigation_type,growth_stage,extreme_event,ndvi,area,previous_year_yield,market_price,profit,yield
5,Pulses,Odisha,Cuttack,Rabi,2015,12.53,24.85,20.52,154.45,68.49,...,108.49,Drip,Maturity,NaN,0.385,3.99,3.45,69128.65,504781.86,2.04
18,Rice,Odisha,Khordha,Zaid,2015,17.06,32.26,24.81,285.17,70.91,...,65.94,Sprinkler,Vegetative,NaN,0.566,6.23,2.33,18603.45,151671.61,1.62
117,Wheat,Odisha,Cuttack,Kharif,2015,10.58,23.29,17.20,67.94,58.79,...,62.52,Rain-fed,Maturity,Flood,0.538,2.68,1.42,24890.56,-8174.94,0.67
215,Maize,Odisha,Khordha,Zaid,2015,10.12,25.57,18.04,189.20,46.62,...,45.57,Rain-fed,Flowering,Flood,0.725,4.96,2.03,18365.35,58417.28,0.91
237,Wheat,Odisha,Puri,Rabi,2015,13.82,26.77,21.07,84.26,52.44,...,38.41,Drip,Vegetative,Flood,0.438,8.75,3.16,21617.72,159185.36,1.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9937,Rice,Odisha,Puri,Kharif,2015,21.07,28.95,25.00,217.97,87.83,...,90.04,Sprinkler,Maturity,Drought,0.711,6.88,2.41,24773.49,158149.40,1.05
9942,Pulses,Odisha,Puri,Zaid,2015,15.07,25.60,22.30,89.50,46.74,...,34.84,Sprinkler,Vegetative,NaN,0.555,0.98,1.63,53758.49,6508.41,1.15
9970,Rice,Odisha,Khordha,Zaid,2015,21.98,35.39,28.07,187.00,85.76,...,105.99,Rain-fed,Maturity,NaN,0.610,6.50,3.36,23629.23,252238.49,1.81
9980,Rice,Odisha,Cuttack,Zaid,2015,24.97,39.07,31.56,254.02,72.95,...,21.59,Rain-fed,Vegetative,NaN,0.822,5.73,4.97,22667.29,271107.46,2.56
